In [18]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [19]:
batch_cfcheck_data = CF_OUTPUTS / "SMOTE" / "rf_SMOTE_simple_highthres_2026-05-05" / "annotated.csv"
batch_cfcheck_df = pd.read_csv(batch_cfcheck_data)

In [20]:
pd.set_option("display.max_rows", None)

In [21]:
batch_cfcheck_df = batch_cfcheck_df.drop(columns=["hltprhc", "target_risk"])

In [22]:

batch_cfcheck_df["cf_id"] = batch_cfcheck_df["cf_id"].replace({"original": "xin"})

batch_cfcheck_df = batch_cfcheck_df.rename(columns={
    "original_risk": "risk_before",
    "predicted_risk" : "predicted_risk_after",
    "meets_target_risk" : "valid",
})

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,valid,risk_before,predicted_risk_after,cf_gen_time_sec,gower_distance
0,0,xin,4.0,3.0,3.0,4.0,2.0,0.0,18.986591,0.0,NaN,0.1667,NaN,2.82,NaN
1,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,17.462750,NaN,False,0.1667,0.1533,NaN,0.9542
2,0,cf_1,NaN,NaN,6.0,NaN,NaN,NaN,18.978740,NaN,False,0.1667,0.0867,NaN,0.8754
3,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,18.766440,5.0,False,0.1667,0.1467,NaN,0.8864


In [23]:
# correct_cf_idx
# Fix cf_id indexing - group by query_index and number counterfactuals sequentially
def fix_cf_id(df):
    # For each query_index group
    fixed_rows = []
    for query_idx, group in df.groupby('query_index', sort=False):
        group = group.copy()
        # First row is xin (original)
        xin_mask = group['cf_id'] == 'xin'
        cf_mask = ~xin_mask

        # Keep xin rows as is
        xin_rows = group[xin_mask].copy()

        # Fix cf rows
        cf_rows = group[cf_mask].copy()
        if len(cf_rows) > 0:
            # Number them cf_1, cf_2, cf_3, etc.
            cf_rows['cf_id'] = [f'cf_{i+1}' for i in range(len(cf_rows))]

        # Combine back
        fixed_rows.append(xin_rows)
        if len(cf_rows) > 0:
            fixed_rows.append(cf_rows)

    return pd.concat(fixed_rows, ignore_index=True)

batch_cfcheck_df = fix_cf_id(batch_cfcheck_df)
batch_cfcheck_df.head(30)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,valid,risk_before,predicted_risk_after,cf_gen_time_sec,gower_distance
0,0,xin,4.0,3.0,3.0,4.0,2.0,0.0,18.986591,0.0,NaN,0.1667,NaN,2.82,NaN
1,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,17.462750,NaN,False,0.1667,0.1533,NaN,0.9542
2,0,cf_2,NaN,NaN,6.0,NaN,NaN,NaN,18.978740,NaN,False,0.1667,0.0867,NaN,0.8754
3,0,cf_3,NaN,NaN,NaN,NaN,NaN,NaN,18.766440,5.0,False,0.1667,0.1467,NaN,0.8864
4,0,cf_4,3.0,NaN,NaN,NaN,NaN,NaN,17.390450,NaN,False,0.1667,0.1733,NaN,0.9580
5,0,cf_5,3.0,NaN,NaN,NaN,NaN,NaN,16.582900,NaN,False,0.1667,0.1867,NaN,1.0000
6,0,cf_6,NaN,NaN,6.0,7.0,NaN,NaN,18.974530,NaN,False,0.1667,0.1833,NaN,0.8756
7,0,cf_7,NaN,NaN,NaN,7.0,NaN,NaN,18.974530,7.0,True,0.1667,0.0567,NaN,0.8756
8,0,cf_8,NaN,NaN,5.0,NaN,NaN,NaN,18.961930,2.0,True,0.1667,0.0367,NaN,0.8763
9,1,xin,3.0,4.0,1.0,2.0,3.0,0.0,22.375680,0.0,NaN,0.5200,NaN,60.85,NaN


In [24]:
# round BMI to 2 decimal places, and set to NaN if equal to xin after rounding
def round_and_clear_bmi(df):
    df = df.copy()

    # Round BMI to 2 decimals for all rows
    df['bmi'] = df['bmi'].round(2)

    # For each query_index, get the xin BMI and compare
    for query_idx in df['query_index'].unique():
        query_mask = df['query_index'] == query_idx
        xin_mask = query_mask & (df['cf_id'] == 'xin')
        cf_mask = query_mask & (df['cf_id'] != 'xin')

        # Get the xin BMI value for this query
        xin_bmi = df.loc[xin_mask, 'bmi'].values[0]

        # Set cf BMI to NaN where it equals xin BMI
        equal_mask = cf_mask & (df['bmi'] == xin_bmi)
        df.loc[equal_mask, 'bmi'] = np.nan

    return df

batch_cfcheck_df = round_and_clear_bmi(batch_cfcheck_df)
batch_cfcheck_df.head(30)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,valid,risk_before,predicted_risk_after,cf_gen_time_sec,gower_distance
0,0,xin,4.0,3.0,3.0,4.0,2.0,0.0,18.99,0.0,NaN,0.1667,NaN,2.82,NaN
1,0,cf_1,NaN,NaN,NaN,NaN,NaN,NaN,17.46,NaN,False,0.1667,0.1533,NaN,0.9542
2,0,cf_2,NaN,NaN,6.0,NaN,NaN,NaN,18.98,NaN,False,0.1667,0.0867,NaN,0.8754
3,0,cf_3,NaN,NaN,NaN,NaN,NaN,NaN,18.77,5.0,False,0.1667,0.1467,NaN,0.8864
4,0,cf_4,3.0,NaN,NaN,NaN,NaN,NaN,17.39,NaN,False,0.1667,0.1733,NaN,0.9580
5,0,cf_5,3.0,NaN,NaN,NaN,NaN,NaN,16.58,NaN,False,0.1667,0.1867,NaN,1.0000
6,0,cf_6,NaN,NaN,6.0,7.0,NaN,NaN,18.97,NaN,False,0.1667,0.1833,NaN,0.8756
7,0,cf_7,NaN,NaN,NaN,7.0,NaN,NaN,18.97,7.0,True,0.1667,0.0567,NaN,0.8756
8,0,cf_8,NaN,NaN,5.0,NaN,NaN,NaN,18.96,2.0,True,0.1667,0.0367,NaN,0.8763
9,1,xin,3.0,4.0,1.0,2.0,3.0,0.0,22.38,0.0,NaN,0.5200,NaN,60.85,NaN


In [25]:
int_columns = [
    "etfruit",
    "eatveg",
    "cgtsmok",
    "alcfreq",
    "slprl",
    "paccnois",
    "dosprt",
]

float_columns = [
    "bmi",
    "risk_before",
    "predicted_risk_after",
]

In [26]:
batch_cfcheck_df[int_columns] = batch_cfcheck_df[int_columns].astype("Int64")

In [27]:
batch_cfcheck_df[float_columns] = batch_cfcheck_df[float_columns].round(2)

batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,valid,risk_before,predicted_risk_after,cf_gen_time_sec,gower_distance
0,0,xin,4,3,3,4,2,0,18.99,0,NaN,0.17,NaN,2.82,NaN
1,0,cf_1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,17.46,<NA>,False,0.17,0.15,NaN,0.9542
2,0,cf_2,<NA>,<NA>,6,<NA>,<NA>,<NA>,18.98,<NA>,False,0.17,0.09,NaN,0.8754
3,0,cf_3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,18.77,5,False,0.17,0.15,NaN,0.8864


In [28]:
batch_cfcheck_df[int_columns] = (
    batch_cfcheck_df[int_columns]
    .astype("string")
    .fillna("")
)

In [29]:
batch_cfcheck_df = batch_cfcheck_df.replace({np.nan : ""})
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,valid,risk_before,predicted_risk_after,cf_gen_time_sec,gower_distance
0,0,xin,4,3,3,4,2,0,18.99,0,,0.17,,2.82,
1,0,cf_1,,,,,,,17.46,,False,0.17,0.15,,0.9542
2,0,cf_2,,,6,,,,18.98,,False,0.17,0.09,,0.8754
3,0,cf_3,,,,,,,18.77,5,False,0.17,0.15,,0.8864


In [30]:
mask = batch_cfcheck_df["cf_id"] == "xin"
batch_cfcheck_df.loc[mask, "risk_before"] = ""
batch_cfcheck_df.head(4)

/tmp/ipykernel_27860/1160248817.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  batch_cfcheck_df.loc[mask, "risk_before"] = ""


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,valid,risk_before,predicted_risk_after,cf_gen_time_sec,gower_distance
0,0,xin,4,3,3,4,2,0,18.99,0,,,,2.82,
1,0,cf_1,,,,,,,17.46,,False,0.17,0.15,,0.9542
2,0,cf_2,,,6,,,,18.98,,False,0.17,0.09,,0.8754
3,0,cf_3,,,,,,,18.77,5,False,0.17,0.15,,0.8864


In [31]:
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# 1. Beräkna Nchanged endast för CF-rader
mask_cf = batch_cfcheck_df["cf_id"] != "xin"

batch_cfcheck_df.loc[mask_cf, "Nchanged"] = (
    batch_cfcheck_df.loc[mask_cf, feature_cols] != ""
).sum(axis=1)

# 2. Konvertera kolumnen till ren string
batch_cfcheck_df["Nchanged"] = batch_cfcheck_df["Nchanged"].astype("string")

# 3. Baseline ska vara tom
batch_cfcheck_df.loc[batch_cfcheck_df["cf_id"] == "xin", "Nchanged"] = ""

In [32]:
batch_cfcheck_df.head(4)

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,valid,risk_before,predicted_risk_after,cf_gen_time_sec,gower_distance,Nchanged
0,0,xin,4,3,3,4,2,0,18.99,0,,,,2.82,,
1,0,cf_1,,,,,,,17.46,,False,0.17,0.15,,0.9542,1
2,0,cf_2,,,6,,,,18.98,,False,0.17,0.09,,0.8754,2
3,0,cf_3,,,,,,,18.77,5,False,0.17,0.15,,0.8864,2


In [33]:
batch_cfcheck_df["Expected"] = ""

In [34]:
order = ["query_index", "cf_id"] + feature_cols + ["cf_gen_time_sec", "gower_distance", "Nchanged", "valid", "risk_before", "predicted_risk_after", "Expected"]

batch_cfcheck_df = batch_cfcheck_df[order]
batch_cfcheck_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,4,3,3,4,2,0,18.99,0,2.82,,,,,,
1,0,cf_1,,,,,,,17.46,,,0.9542,1,False,0.17,0.15,
2,0,cf_2,,,6,,,,18.98,,,0.8754,2,False,0.17,0.09,
3,0,cf_3,,,,,,,18.77,5,,0.8864,2,False,0.17,0.15,
4,0,cf_4,3,,,,,,17.39,,,0.958,2,False,0.17,0.17,
5,0,cf_5,3,,,,,,16.58,,,1.0,2,False,0.17,0.19,
6,0,cf_6,,,6,7,,,18.97,,,0.8756,3,False,0.17,0.18,
7,0,cf_7,,,,7,,,18.97,7,,0.8756,3,True,0.17,0.06,
8,0,cf_8,,,5,,,,18.96,2,,0.8763,3,True,0.17,0.04,
9,1,xin,3,4,1,2,3,0,22.38,0,60.85,,,,,,


# picking correct CF

### expected


In [35]:
# Check directional constraints directly
# These features should only improve (increase) or stay the same
SHOULD_INCREASE = ["cgtsmok", "alcfreq", "dosprt"]

# These features should only improve (decrease) or stay the same
SHOULD_DECREASE = ["bmi", "etfruit", "eatveg", "slprl", "paccnois"]


print("Directional constraints:")
print(f"  Should increase (↑): {SHOULD_INCREASE}")
print(f"  Should decrease (↓): {SHOULD_DECREASE}")


Directional constraints:
  Should increase (↑): ['cgtsmok', 'alcfreq', 'dosprt']
  Should decrease (↓): ['bmi', 'etfruit', 'eatveg', 'slprl', 'paccnois']


In [36]:
# Check if CFs violate directional constraints
# First, ensure numeric columns are actually numeric for comparison
numeric_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# Create a working copy with numeric values
df_check = batch_cfcheck_df.copy()
for col in numeric_cols:
    df_check[col] = pd.to_numeric(df_check[col], errors="coerce")

# Extract baseline values per query_index
baseline_values = df_check[df_check["cf_id"] == "xin"].set_index("query_index")[numeric_cols]

# Check violations for each CF
def check_violations(row):
    if row["cf_id"] == "xin":
        return ""

    baseline = baseline_values.loc[row["query_index"]]
    violations = []

    # Check features that should increase
    for feat in SHOULD_INCREASE:
        if pd.notna(row[feat]) and pd.notna(baseline[feat]):
            if row[feat] < baseline[feat]:
                violations.append(f"{feat}↓")

    # Check features that should decrease
    for feat in SHOULD_DECREASE:
        if pd.notna(row[feat]) and pd.notna(baseline[feat]):
            if row[feat] > baseline[feat]:
                violations.append(f"{feat}↑")

    return "NO: " + ", ".join(violations) if violations else True

batch_cfcheck_df["Expected"] = df_check.apply(check_violations, axis=1)
batch_cfcheck_df.head(10)


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,4,3,3,4,2,0,18.99,0,2.82,,,,,,
1,0,cf_1,,,,,,,17.46,,,0.9542,1,False,0.17,0.15,True
2,0,cf_2,,,6,,,,18.98,,,0.8754,2,False,0.17,0.09,True
3,0,cf_3,,,,,,,18.77,5,,0.8864,2,False,0.17,0.15,True
4,0,cf_4,3,,,,,,17.39,,,0.958,2,False,0.17,0.17,True
5,0,cf_5,3,,,,,,16.58,,,1.0,2,False,0.17,0.19,True
6,0,cf_6,,,6,7,,,18.97,,,0.8756,3,False,0.17,0.18,True
7,0,cf_7,,,,7,,,18.97,7,,0.8756,3,True,0.17,0.06,True
8,0,cf_8,,,5,,,,18.96,2,,0.8763,3,True,0.17,0.04,True
9,1,xin,3,4,1,2,3,0,22.38,0,60.85,,,,,,


### is valid

In [37]:
# --- Select exactly one CF per query_index (random valid without violations) ---

# Split baseline and CF rows
df_xin = batch_cfcheck_df[batch_cfcheck_df["cf_id"] == "xin"]
df_cf  = batch_cfcheck_df[batch_cfcheck_df["cf_id"] != "xin"]

def select_random_cf(group):
    """Select one random CF that is valid and has no violations (Expected is empty)."""
    # First try: valid AND no violations
    good_cfs = group[(group["valid"] == True) & (group["Expected"] == "")]
    if len(good_cfs) > 0:
        return good_cfs.sample(n=1, random_state=42)

    # Fallback: any valid CF
    valid_cfs = group[group["valid"] == True]
    if len(valid_cfs) > 0:
        return valid_cfs.sample(n=1, random_state=42)

    # Last resort: first CF
    return group.iloc[[0]]

# Select one random CF per query_index
df_cf_selected = df_cf.groupby("query_index", group_keys=False).apply(select_random_cf).reset_index(drop=True)

# Combine baseline + selected CF
batch_cfcheck_df = pd.concat([df_xin, df_cf_selected], ignore_index=True)

# Ensure xin appears before CF for each query_index
batch_cfcheck_df["is_xin"] = (batch_cfcheck_df["cf_id"] == "xin").astype(int)
batch_cfcheck_df = (
    batch_cfcheck_df
    .sort_values(["query_index", "is_xin"], ascending=[True, False])
    .drop(columns="is_xin")
)
batch_cfcheck_df

/tmp/ipykernel_27860/2325069138.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cf_selected = df_cf.groupby("query_index", group_keys=False).apply(select_random_cf).reset_index(drop=True)


,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after,Expected
0,0,xin,4,3,3,4,2,0,18.99,0,2.82,,,,,,
9,0,cf_8,,,5,,,,18.96,2,,0.8763,3,True,0.17,0.04,True
1,1,xin,3,4,1,2,3,0,22.38,0,60.85,,,,,,
10,1,cf_1,,,,,1,,,,,0.8796,1,True,0.52,0.06,True
2,2,xin,5,3,1,1,4,0,22.69,7,3.14,,,,,,
11,2,cf_1,3,,,,2,,22.68,,,1.0,3,True,0.33,0.06,True
3,3,xin,3,3,6,6,2,0,24.34,1,3.31,,,,,,
12,3,cf_1,,,,,,,,,,0.8753,0,False,0.04,0.03,True
4,4,xin,5,4,2,7,1,0,21.26,3,3.38,,,,,,
13,4,cf_2,,,6,,,,,,,0.9868,1,True,0.01,0.0,True


In [38]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_cfcheck_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.99
  dosprt: 0


=== Counterfactuals ===

--- cf_8 ---
Predicted risk: 0.04
Valid: True
Changes:
  cgtsmok: 3 → 5
  bmi: 18.99 → 18.96
  dosprt: 0 → 2


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.38
  dosprt: 0


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.06
Valid: True
Changes:
  slprl: 3 → 1


=== Query index '2' ===
Task / Target: hltprhc
Query instance index: 2

Original instance:
  etfruit: 5
  eatveg: 3
  cgtsmok: 1
  alcfreq: 1
  slprl: 4
  paccnois: 0
  bmi: 22.69
  dosprt: 7


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.06
Valid: True
Changes:
  etfruit: 5 → 3
  slprl: 4 → 2
  bmi: 22.69 → 22.68


=== Query index '3' ===
Task / Target: hltprhc
Qu